In [1]:
import pandas as pd
import numpy as np

In [3]:
sales = pd.read_csv("../data/cleaned/sales_daily_clean.csv")
sku = pd.read_csv("../data/cleaned/sku_master_clean.csv")
calendar = pd.read_csv("../data/cleaned/calendar_clean.csv")
inventory = pd.read_csv("../data/cleaned/inventory_snapshots_clean.csv")

In [4]:
sales["Date"] = pd.to_datetime(sales["Date"])
sku["Launch_Date"] = pd.to_datetime(sku["Launch_Date"])
inventory["Snapshot_Date"] = pd.to_datetime(inventory["Snapshot_Date"])
calendar["date"] = pd.to_datetime(calendar["date"])

#### Sales

In [5]:
model_data = sales.copy()

In [7]:
print(model_data.shape)
model_data.head()

(36550, 6)


,Date,SKU,Units_Sold,Revenue,Price,Promotion
0,2024-01-01,SKU001,5,18320.25,3664.05,0
1,2024-01-01,SKU002,15,57085.35,3805.69,0
2,2024-01-01,SKU003,5,40391.15,8078.23,0
3,2024-01-01,SKU004,5,34307.85,6861.57,0
4,2024-01-01,SKU005,12,113918.64,9493.22,0


In [8]:
model_data = model_data.merge(
    calendar[
        ["date", "year", "month", "quarter", "week", "day_of_week", "is_weekend", "season", "is_holiday"]],
    left_on="Date", right_on="date", how="left" )

In [11]:
model_data.drop(columns=["date"], inplace=True)

In [12]:
print(model_data.shape)
model_data.head()

(36550, 14)


,Date,SKU,Units_Sold,Revenue,Price,Promotion,year,month,quarter,week,day_of_week,is_weekend,season,is_holiday
0,2024-01-01,SKU001,5,18320.25,3664.05,0,2024,1,Q1,1,Monday,0,Winter,0
1,2024-01-01,SKU002,15,57085.35,3805.69,0,2024,1,Q1,1,Monday,0,Winter,0
2,2024-01-01,SKU003,5,40391.15,8078.23,0,2024,1,Q1,1,Monday,0,Winter,0
3,2024-01-01,SKU004,5,34307.85,6861.57,0,2024,1,Q1,1,Monday,0,Winter,0
4,2024-01-01,SKU005,12,113918.64,9493.22,0,2024,1,Q1,1,Monday,0,Winter,0


#### SKU/Product Features

In [13]:
model_data = model_data.merge(
    sku[["SKU", "Category", "Subcategory", "Cost_Price", "Selling_Price", "Gross_Margin_Per_Unit", "Negative_Margin"]],
    on="SKU",
    how="left"
)

In [14]:
print(model_data.shape)
model_data.head()

(36550, 20)


,Date,SKU,Units_Sold,Revenue,Price,Promotion,year,month,quarter,week,day_of_week,is_weekend,season,is_holiday,Category,Subcategory,Cost_Price,Selling_Price,Gross_Margin_Per_Unit,Negative_Margin
0,2024-01-01,SKU001,5,18320.25,3664.05,0,2024,1,Q1,1,Monday,0,Winter,0,Furniture,Chair,1758.45,3664.05,1905.60,False
1,2024-01-01,SKU002,15,57085.35,3805.69,0,2024,1,Q1,1,Monday,0,Winter,0,Home Decor,Table,3867.09,3805.69,-61.40,True
2,2024-01-01,SKU003,5,40391.15,8078.23,0,2024,1,Q1,1,Monday,0,Winter,0,Kitchen,Cushion,589.48,8078.23,7488.75,False
3,2024-01-01,SKU004,5,34307.85,6861.57,0,2024,1,Q1,1,Monday,0,Winter,0,Lighting,Cookware,1445.74,6861.57,5415.83,False
4,2024-01-01,SKU005,12,113918.64,9493.22,0,2024,1,Q1,1,Monday,0,Winter,0,Storage,Lamp,5543.63,9493.22,3949.59,False


In [15]:
model_data = model_data.sort_values(["SKU", "Date"]).reset_index(drop=True)

In [21]:
model_data[["SKU", "Date", "Units_Sold"]].head(20)

,SKU,Date,Units_Sold
0,SKU001,2024-01-01,5
1,SKU001,2024-01-02,13
2,SKU001,2024-01-03,12
3,SKU001,2024-01-04,23
4,SKU001,2024-01-05,16
5,SKU001,2024-01-06,18
6,SKU001,2024-01-07,19
7,SKU001,2024-01-08,12
8,SKU001,2024-01-09,10
9,SKU001,2024-01-10,11


#### Lag Features

In [22]:
model_data["Lag_1"] = (model_data.groupby("SKU")["Units_Sold"].shift(1))

In [23]:
model_data["Lag_7"] = (model_data.groupby("SKU")["Units_Sold"].shift(7))

In [24]:
model_data["Lag_14"] = (model_data.groupby("SKU")["Units_Sold"].shift(14))

In [25]:
model_data["Lag_30"] = (model_data.groupby("SKU")["Units_Sold"].shift(30))

In [26]:
model_data[["SKU", "Date", "Units_Sold", "Lag_1", "Lag_7", "Lag_14", "Lag_30"]].head(40)

,SKU,Date,Units_Sold,Lag_1,Lag_7,Lag_14,Lag_30
0,SKU001,2024-01-01,5,NaN,NaN,NaN,NaN
1,SKU001,2024-01-02,13,5.0,NaN,NaN,NaN
2,SKU001,2024-01-03,12,13.0,NaN,NaN,NaN
3,SKU001,2024-01-04,23,12.0,NaN,NaN,NaN
4,SKU001,2024-01-05,16,23.0,NaN,NaN,NaN
5,SKU001,2024-01-06,18,16.0,NaN,NaN,NaN
6,SKU001,2024-01-07,19,18.0,NaN,NaN,NaN
7,SKU001,2024-01-08,12,19.0,5.0,NaN,NaN
8,SKU001,2024-01-09,10,12.0,13.0,NaN,NaN
9,SKU001,2024-01-10,11,10.0,12.0,NaN,NaN


#### Rolling Features

In [27]:
model_data["Rolling_Mean_7"] = (model_data.groupby("SKU")["Units_Sold"].transform(lambda x: x.shift(1).rolling(7).mean()))

In [29]:
model_data["Rolling_Mean_14"] = (model_data.groupby("SKU")["Units_Sold"].transform(lambda x: x.shift(1).rolling(14).mean()))

In [30]:
model_data["Rolling_Mean_30"] = (model_data.groupby("SKU")["Units_Sold"].transform(lambda x: x.shift(1).rolling(30).mean()))

#### Demand Trend Feature

In [31]:
model_data["Demand_Trend"] = (
    model_data["Rolling_Mean_7"] /
    model_data["Rolling_Mean_30"]
)

#### Price Features

In [32]:
model_data["Price_Difference"] = (
    model_data["Price"] -
    model_data["Selling_Price"]
)

In [33]:
model_data["Price_Discount_Pct"] = (
    (model_data["Selling_Price"] - model_data["Price"])
    / model_data["Selling_Price"]
) * 100

#### Inventory Features

In [70]:
inventory_features = inventory[
    [
        "Snapshot_Date",
        "SKU",
        "Current_Stock",
        "On_Order",
        "Lead_Time_Days",
        "Safety_Stock",
        "Reorder_Point",
        "Inventory_Value"
    ]
].copy()

In [71]:
inventory_features = inventory_features.rename(
    columns={"Snapshot_Date": "Date"}
)

In [72]:
model_data = model_data.sort_values(
    ["SKU", "Date"]
).reset_index(drop=True)

inventory_features = inventory_features.sort_values(
    ["SKU", "Date"]
).reset_index(drop=True)

In [73]:
model_data = pd.merge_asof(
    model_data.sort_values("Date"),
    inventory_features.sort_values("Date"),
    on="Date",
    by="SKU",
    direction="backward"
)

In [83]:
model_data[
    [
        "Date",
        "SKU",
        "Units_Sold",
        "Current_Stock",
        "On_Order",
        "Safety_Stock",
        "Reorder_Point"
    ]
].head()

,Date,SKU,Units_Sold,Current_Stock,On_Order,Safety_Stock,Reorder_Point
0,2024-01-01,SKU001,5,16,23,7,18
1,2024-01-01,SKU007,27,12,2,7,10
2,2024-01-01,SKU032,7,19,21,5,14
3,2024-01-01,SKU033,15,10,17,5,10
4,2024-01-01,SKU034,17,29,2,5,11


In [75]:
model_data["Stock_Gap"] = (
    model_data["Current_Stock"]
    - model_data["Reorder_Point"]
)

In [76]:
model_data["Below_Reorder_Point"] = (
    model_data["Current_Stock"]
    < model_data["Reorder_Point"]
).astype(int)

In [77]:
model_data["Below_Safety_Stock"] = (
    model_data["Current_Stock"]
    < model_data["Safety_Stock"]
).astype(int)

In [78]:
model_data["Stock_Coverage_Days"] = (
    model_data["Current_Stock"]
    / model_data["Rolling_Mean_7"].replace(0, np.nan)
)

In [79]:
model_data.isnull().sum()

Date                        0
SKU                         0
Units_Sold                  0
Revenue                     0
Price                       0
Promotion                   0
year                        0
month                       0
quarter                     0
week                        0
day_of_week                 0
is_weekend                  0
season                      0
is_holiday                  0
Category                    0
Subcategory                 0
Cost_Price                  0
Selling_Price               0
Gross_Margin_Per_Unit       0
Negative_Margin             0
Lag_1                      50
Lag_7                     350
Lag_14                    700
Lag_30                   1500
Rolling_Mean_7            350
Rolling_Mean_30          1500
Rolling_Mean_14           700
Demand_Trend             1500
Price_Difference            0
Price_Discount_Pct          0
Current_Stock_x             0
On_Order_x                  0
Lead_Time_Days_x            0
Safety_Sto

In [80]:
print(model_data.shape)
print(model_data["SKU"].nunique())
print(model_data["Date"].min())
print(model_data["Date"].max())

(36550, 52)
50
2024-01-01 00:00:00
2025-12-31 00:00:00


In [81]:
model_data.isnull().sum()

Date                        0
SKU                         0
Units_Sold                  0
Revenue                     0
Price                       0
Promotion                   0
year                        0
month                       0
quarter                     0
week                        0
day_of_week                 0
is_weekend                  0
season                      0
is_holiday                  0
Category                    0
Subcategory                 0
Cost_Price                  0
Selling_Price               0
Gross_Margin_Per_Unit       0
Negative_Margin             0
Lag_1                      50
Lag_7                     350
Lag_14                    700
Lag_30                   1500
Rolling_Mean_7            350
Rolling_Mean_30          1500
Rolling_Mean_14           700
Demand_Trend             1500
Price_Difference            0
Price_Discount_Pct          0
Current_Stock_x             0
On_Order_x                  0
Lead_Time_Days_x            0
Safety_Sto

In [85]:
model_data.to_csv("../data/cleaned/feature_engineered.csv",index=False)